<div style='text-align:center; border:1px solid #cecece; padding:1rem'>
    <h1>Convenções de Contagem de Tempo na Matemática Financeira</h1>
    <h3>Juros Exatos, Juros Comerciais e Juros Bancários</h3>
</div>

## Bloco 1: Apresentação

### Descrição
Este notebook explora as **três principais convenções de contagem de tempo** utilizadas na matemática financeira para converter prazos dados em dias para uma unidade coerente com a taxa de juros (geralmente anual). A escolha da convenção impacta diretamente o valor final dos juros, pois altera a forma como o período ($n$) é calculado.

---

### Objetivo
- Compreender as diferenças entre Juros Exatos, Comerciais e Bancários
- Implementar funções em Python para calcular juros sob cada convenção
- Comparar os resultados obtidos pelas três modalidades
- Visualizar graficamente o impacto de cada convenção no valor dos juros

---

### Autoria
Wellington M. Santos - Cientista de Dados  
**LinkedIn**: [in/wellington-moreira-santos](https://linkedin.com/in/wellington-moreira-santos)  
**GitHub**: [/esscova](https://www.github.com/esscova)  
**Email**: [wsantos08@hotmail.com](mailto:wsantos08@hotmail.com)

---

### Sumário do Conteúdo
1. Introdução e Conceitos Fundamentais
2. Implementação da Convenção de Juros Exatos
3. Implementação da Convenção de Juros Comerciais
4. Implementação da Convenção de Juros Bancários
5. Comparação Prática entre as Três Convenções
6. Visualização Gráfica dos Resultados
7. Resumo e Considerações Finais

## Bloco 2: Introdução e Conceitos Fundamentais

### 1. Introdução e Conceitos Fundamentais

A fórmula básica do juros simples é:

$$
J = C \times i \times n
$$

Onde:
- $J$ = Juros
- $C$ = Capital Inicial
- $i$ = Taxa de juros (na mesma unidade de tempo de $n$)
- $n$ = Período de tempo

O problema: Quando o prazo é dado em dias e a taxa é anual, precisamos converter $n$ para anos. A convenção escolhida defini como essa convenção é feita.

### Bibliotecas Necessárias

In [1]:
# dependencias

# data 
import pandas as pd
import numpy as np

# data viz
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
import calendar

print('=== Bibliotecas importadas com sucesso ===')

=== Bibliotecas importadas com sucesso ===


In [2]:
# configurações
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12,6)
plt.rcParams['font.size'] = 11

print('=== Configurações aplicadas com sucesso ===')

=== Configurações aplicadas com sucesso ===


## Bloco 3: Juros Exatos

**Características:**
- **Meses:** Cada mês é contado com seu número real de dias (ex: marco = 31, abril = 30, fevereiro = 28 ou 29)
- **Ano:** Considerado com **365 dias** (ou 366, se for bissexto)
- **Aplicação:** Forma mais precisa matematicamente, reflete o tempo cronológico real

**Fórmula de conversão**
$$
n = \frac{\text{dias reais entre as datas}}{365 \ \text{(ou 366 se bissexto)}}
$$

Assimilando ao Python, temos:

In [11]:
def calcular_juros_exatos(capital, taxa_anual, data_inicio, data_fim):
    """
    Calcula juros utilizando a convenção EXATA.

    Args:
        capital    : Valor inicial (R$)
        taxa_anual : Taxa de juros anual (decimal, ex: 0.10 para 10%)
        data_inicio: Data inicial (datetime ou string 'YYYY-MM-DD')
        data_fim   : Data final (datetime ou string 'YYYY-MM-DD')

    Returns:
        dict com dias_reais, dias_ano, prazo_anos, juros, montante
    """

    # preparacao de strings para datetime
    if isinstance(data_inicio, str):
        data_inicio = datetime.strptime(data_inicio, '%Y-%m-%d')
    if isinstance(data_fim, str):
        data_fim = datetime.strptime(data_fim, '%Y-%m-%d')

    # contagem exata de dias
    dias_reais = (data_fim - data_inicio).days

    # verificar bissexto :: considera ano de inicio
    ano = data_inicio.year
    eh_bissexto = calendar.isleap(ano)
    dias_no_ano = 366 if eh_bissexto else 365

    # calculo prazo em anos :: convencao exata
    prazo_anos = dias_reais / dias_no_ano

    # calculo juros
    juros = capital * taxa_anual * prazo_anos
    montante = capital + juros

    return {
        'convenção'   : 'Juros Exatos',
        'capital'     : capital,
        'taxa_anual'  : taxa_anual,
        'data_inicio' : data_inicio.strftime('%Y-%m-%d'),
        'data_fim'    : data_fim.strftime('%Y-%m-%d'),
        'dias_reais'  : dias_reais,
        'dias_no_ano' : dias_no_ano,
        'ano_bissexto': eh_bissexto,
        'prazo_anos'  : round(prazo_anos, 10),
        'juros'       : round(juros, 2),
        'montante'    : round(montante, 2)
    }

**Exemplo:** Empréstimo de R\\$ 10.000, taxa 10% a.a., de 01/01/2026 a 31/12/2026

In [15]:
print('=' * 50)
print('JUROS EXATOS')
print('=' * 50)

res = calcular_juros_exatos(
    capital     = 10_000,
    taxa_anual  = 10/100, # 0.10
    data_inicio = '2026-01-01',
    data_fim    = '2026-12-31'
)

for k,v in res.items():
    print(f'{k:20s}: {v}')

JUROS EXATOS
convenção           : Juros Exatos
capital             : 10000
taxa_anual          : 0.1
data_inicio         : 2026-01-01
data_fim            : 2026-12-31
dias_reais          : 364
dias_no_ano         : 365
ano_bissexto        : False
prazo_anos          : 0.997260274
juros               : 997.26
montante            : 10997.26


## Bloco 4: Juros Comerciais

**Características:**

- **Meses:** Todos os meses travados em **30 dias**, independente do mês.
- **Ano:** Sempre **360 dias** ($12 \times 30$)

**Fórmula de conversão:**
$$
n = \frac{\text{dias informados}}{360}
$$

> Note que aqui não contamos dias reais do calendário, mas sim o número de dias informado no contrato/operação, ou calculado pela regra comercial.

Vamos ao Python:

In [3]:
def calcular_juros_comerciais(capital, taxa_anual, dias):
    """
    Calcula juros utilizando a convenção COMERCIAL.

    Args:
        - capital   : Valor inicial (R$)
        - taxa_anual: Taxa de juros anual (decimal)
        - dias      : Número de dias do prazo (pode ser calculado pela regra comercial)
    
    Return:
        - dict com dias, dias_ano, prazo_anos, juros, montante
    """

    # ano comercial sempre tem 360 dias
    dias_no_ano = 360

    # calculdo do prazo em anos :: convencao comercial
    prazo_anos = dias / dias_no_ano

    # calculo juros
    juros    = capital * taxa_anual * prazo_anos
    montante = capital + juros

    return {
        'convenção'       : 'juros Comerciais',
        'capital'         : capital,
        'taxa_anual'      : taxa_anual,
        'dias_informados' : dias,
        'dias_no_ano'     : dias_no_ano,
        'prazo_anos'      : round(prazo_anos, 10),
        'juros'           : round(juros, 2),
        'montante'        : round(montante, 2),
    }

In [4]:
def dias_comerciais(data_inicio, data_fim):
    """
    Calcula o prazo em dias pela regra comercial (mês = 20 dias).
    """
    if isinstance(data_inicio, str):
        data_inicio = datetime.strptime(data_inicio, '%Y-%m-%d')
    if isinstance(data_fim, str):
        data_fim = datetime.strptime(data_fim, '%Y-%m-%d')

    # diferenca
    anos  = data_fim.year - data_inicio.year
    meses = data_fim.month - data_inicio.month
    dias  = data_fim.day - data_inicio.day

    # ajuste se necessario
    total_meses = anos * 12 + meses
    if dias < 0:
        total_meses -= 1
        dias += 30

    total_dias_comercial = total_meses * 30 + dias

    return total_dias_comercial

A seguir, usando o mesmo exemplo anterior - Empréstimo de R$ 10.000, taxa 10% a.a., de 01/01/2026 a 31/12/2026 - vemos:

In [5]:
print('=' * 50)
print('JUROS COMERCIAIS')
print('=' * 50)

dias_com = dias_comerciais('2026-01-01', '2026-12-31')
print(f'Dias pelo critério comercial: {dias_com}')

res_comercial = calcular_juros_comerciais(
    capital    = 10_000,
    taxa_anual = 0.10,
    dias       = dias_com
)

for k, v in res_comercial.items():
    print(f'{k:20s}: {v}')

JUROS COMERCIAIS
Dias pelo critério comercial: 360
convenção           : juros Comerciais
capital             : 10000
taxa_anual          : 0.1
dias_informados     : 360
dias_no_ano         : 360
prazo_anos          : 1.0
juros               : 1000.0
montante            : 11000.0


## Bloco 5: Juros Bancários
Combina as duas convenções anteriores, desenhada para maximizar o juro recebido.

**Características:**
- **Prazo:** Contagem exata do calendário (maximiza dias)
- **Ano:** Ano comercial de 360 dias (minimiza o divisor)

**Fórmula de convenção:**
$$
n = \frac{\text{dias reais entre as datas}}{360}
$$

> Essa convenção é favorável ao credor (quem recebe o juros) e desfavorável ao devedor.

Vemos em Python:

In [6]:
def calcular_juros_bancarios(capital, taxa_anual, data_inicio, data_fim):
    """
    Calcula juros utilizando a convenção BANCÁRIA.
    
    Parâmetros:
    - capital: Valor inicial (R$)
    - taxa_anual: Taxa de juros anual (decimal)
    - data_inicio: Data inicial (datetime ou string)
    - data_fim: Data final (datetime ou string)
    
    Retorna:
    - dict com dias_reais, dias_ano, prazo_anos, juros, montante
    """
    # converter strings para datetime se necessario
    if isinstance(data_inicio, str):
        data_inicio = datetime.strptime(data_inicio, '%Y-%m-%d')
    if isinstance(data_fim, str):
        data_fim = datetime.strptime(data_fim, '%Y-%m-%d')
    
    # numerador :: dias REAIS do calendário
    dias_reais = (data_fim - data_inicio).days
    
    # denominador :: ano COMERCIAL de 360 dias
    dias_no_ano = 360
    
    # calculo do prazo em anos (convenção bancária)
    prazo_anos = dias_reais / dias_no_ano
    
    # calculo dos juros
    juros = capital * taxa_anual * prazo_anos
    montante = capital + juros
    
    return {
        'convenção'  : 'Juros Bancários',
        'capital'    : capital,
        'taxa_anual' : taxa_anual,
        'data_inicio': data_inicio.strftime('%Y-%m-%d'),
        'data_fim'   : data_fim.strftime('%Y-%m-%d'),
        'dias_reais' : dias_reais,
        'dias_no_ano': dias_no_ano,
        'prazo_anos' : round(prazo_anos, 10),
        'juros'      : round(juros, 2),
        'montante'   : round(montante, 2)
    }

Seguindo com o exemplo:

In [7]:
print('=' * 50)
print('JUROS BANCÁRIOS')
print('=' * 50)

res_bancario = calcular_juros_bancarios(
    capital=10_000,
    taxa_anual=0.10,
    data_inicio='2024-01-01',
    data_fim='2024-12-31'
)

for k, v in res_bancario.items():
    print(f"{k:20s}: {v}")

JUROS BANCÁRIOS
convenção           : Juros Bancários
capital             : 10000
taxa_anual          : 0.1
data_inicio         : 2024-01-01
data_fim            : 2024-12-31
dias_reais          : 365
dias_no_ano         : 360
prazo_anos          : 1.0138888889
juros               : 1013.89
montante            : 11013.89
